# 📘 [강의 자료] 초보자를 위한 딥러닝 실주행시간 예측 모델 구축하기

이 주피터 노트북은 딥러닝의 복잡한 용어와 개념을 **실제 비유와 예시**를 통해 쉽고 직관적으로 이해하고, **실주행시간(Time_Driving) 예측 딥러닝 모델**을 직접 구현해 볼 수 있도록 구성된 강의 자료입니다.

---

## 1. 딥러닝 개념, 한눈에 이해하기 (수능 공부 비유)

딥러닝 모델을 만드는 과정은 **'시험을 준비하는 학생을 가르치는 과정'**과 똑같습니다.

| 딥러닝 용어 | 쉬운 비유 | 이 모델(실주행시간 예측)에서의 실제 의미 |
| :--- | :--- | :--- |
| **Epoch (에포크)** | **문제집 전체를 1번 풀기** | 전체 주행 데이터(X_train)를 처음부터 끝까지 1번 공부하는 것입니다. `epochs=50`은 문제집을 총 50번 반복해서 회독한다는 뜻입니다. |
| **Batch Size (배치 크기)** | **한 번에 풀고 채점하는 문제 수** | `batch_size=128`은 **128개의 주행 데이터를 모아서 풀고 바로 정답을 확인(채점)**한 뒤 다음 128문제를 푸는 방식입니다. |
| **Loss (손실/오차)** | **시험 채점 후 틀린 오차 점수** | AI가 주행시간을 **40분**으로 예측했는데 실주행시간이 **50분**이었다면 오차는 **10분**입니다. `mean_squared_error`는 이 오차들을 제곱해서 평균을 낸 채점 점수이며, 0에 가까울수록 정답에 가깝습니다. |
| **Optimizer (최적화 도구)** | **오답노트 가르치는 과외 선생님** | 채점 결과(Loss)를 보고 "퇴근시간대 가중치를 올려야겠구나"라며 **다음엔 안 틀리도록 뇌(신경망 가중치)를 수정해 주는 선생님**입니다. `adam`은 가장 성적을 잘 올려주는 족집게 선생님입니다. |
| **EarlyStopping (조기 종료)** | **성적이 안 오를 때 공부를 멈추기** | 50번 공부하기로 했지만, 9번 연속(`patience=9`)으로 모의고사 오차(`val_loss`)가 줄어들지 않으면 "더 풀어봤자 외우기만 한다"며 학습을 자동으로 중단합니다. |
| **Activation (활성화 함수)** | **공부할 때 멘탈 관리 방식** | **SELU**는 실제 텐서플로에 존재하는 공식 함수로, 학습 중 데이터가 튀지 않게 **스스로 마음을 다잡고 페이스를 유지(자체 정규화)**해 주는 똑똑한 함수입니다. |
| **Dropout (드롭아웃)** | **일부러 암기 방지 훈련하기** | `Dropout(0.1)`은 공부할 때 10%의 신경망 노드를 쉬게 만들어서, 특정 문제에만 너무 똑같이 의존하는 **과적합(Overfitting)**을 예방합니다. |


In [ ]:
# ==========================================
# 2. 딥러닝 모델 구현 코드
# ==========================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1) 모델 구조 상자 만들기 (Sequential)
model = Sequential()

# 2) 첫 번째 은닉층: 64개 노드, SELU 활성화 함수
# input_dim은 입력 데이터의 특징(Feature) 개수입니다.
model.add(Dense(64, activation='selu', input_dim=X.shape[1]))

# 3) 드롭아웃 레이어: 과적합 방지를 위해 10% 노드 비활성화
model.add(Dropout(0.1))

# 4) 두 번째 은닉층: 32개 노드, SELU 활성화 함수
model.add(Dense(32, activation='selu'))

# 5) 세 번째 은닉층: 16개 노드, SELU 활성화 함수
model.add(Dense(16, activation='selu'))

# 6) 출력층: 실주행시간 1개의 수치를 예측하므로 노드 수 1개, linear 활성화 함수 사용
model.add(Dense(1, activation='linear'))

# 7) 조기 종료(EarlyStopping) 설정
# 검증 오차(val_loss)를 모니터링하며, 9번의 epoch 동안 개선이 없으면 중지합니다.
estop = EarlyStopping(monitor='val_loss', patience=9)

# 8) 모델 컴파일 (최적화 도구, 손실 함수, 평가 지표 설정)
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mse']
)

# 9) 모델 학습 수행
# batch_size=128, epochs=50으로 설정하여 학습을 진행하고 결과를 history 변수에 저장합니다.
history = model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_data=(X_valid, y_valid),
    callbacks=[estop]
)


---

## 코드 핵심 라인별 쉬운 요약

| 코드 구문 | 초보자용 핵심 해설 |
| :--- | :--- |
| `Sequential()` | 레이어를 순서대로 차곡차곡 쌓아 올리는 상자를 준비합니다. |
| `Dense(64, activation='selu', input_dim=...)` | 첫 입구에서 64개 노드로 데이터 특징을 수집하고, SELU로 페이스를 조절합니다. |
| `Dropout(0.1)` | 10% 노드를 무작위로 쉬게 하여 암기식 공부(과적합)를 예방합니다. |
| `Dense(1, activation='linear')` | 최종 결과물인 주행시간 숫자 1개를 변형 없이 그대로 출력합니다. |
| `EarlyStopping(..., patience=9)` | 모니터링 지표(`val_loss`)가 9번 동안 안 나아지면 자동으로 학습을 멈춥니다. |
| `model.compile(...)` | 채점 방식(`mean_squared_error`)과 과외 선생님(`adam`)을 확정합니다. |
| `model.fit(...)` | 128개씩 묶어서 풀며 최대 50번 문제집을 반복 학습합니다. |


In [ ]:
from tensorflow.keras.utils import plot_model

# 모델 구조 시각화 및 이미지 파일 저장
plot_model(
    model,
    to_file='model_architecture.png',  # 저장할 파일명
    show_shapes=True,                  # 데이터 모양(Shape) 표시 여부
    show_layer_names=True,             # 레이어 이름 표시 여부
    show_layer_activations=True        # 활성화 함수(selu, linear 등) 표시 여부
)

# 주피터 노트북 내 화면에 직접 표시하고 싶을 때
from IPython.display import Image
Image('model_architecture.png')

# 📘 [강의 자료] 딥러닝 모델의 학습 경과 시각화 및 일반화 성능 평가

이번 시간에는 앞서 만든 딥러닝 모델이 공부를 잘하고 있는지, 혹시 문제집만 외우는 **과적합(Overfitting)** 상태에 빠지지는 않았는지 평가하는 방법을 배웁니다.

---

## 학습 MSE vs 검증 MSE 이해하기

모델 학습의 목표는 단순 문제집 암기가 아니라 **'처음 보는 새로운 도로 상황에서도 주행시간을 잘 예측하는 것(일반화 성능)'**입니다.

| 구분 | 코드명 | 쉬운 비유 | 설명 |
| :--- | :--- | :--- | :--- |
| **학습 MSE** | `history.history['mse']` | **연습 문제 집 오차 점수** | 모델이 직접 학습에 사용한 데이터(`X_train`)를 풀며 발생한 오차입니다. |
| **검증 MSE** | `history.history['val_mse']` | **실전 모의고사 오차 점수** | 모델 학습에는 쓰이지 않고 오직 성능 평가용으로만 떼어둔 데이터(`X_valid`)로 테스트한 오차입니다. |

### 💡 그래프 해석 요령
* **이상적인 상태:** 에포크가 진행됨에 따라 `mse`와 `val_mse`가 함께 안정적으로 낮아집니다.
* **과적합(Overfitting) 발생:** `mse`는 계속 떨어지는데, `val_mse`가 다시 올라가기 시작한다면 문제집 답만 외우기 시작한 것입니다.


In [ ]:
# ==========================================
# 2. 모델 성능 시각화 코드 (문제 답안)
# ==========================================

import matplotlib.pyplot as plt

# 1) 1개의 그래프 선에 학습 MSE와 검증 MSE를 순서대로 추가
plt.plot(history.history['mse'])
plt.plot(history.history['val_mse'])

# 2) 그래프 제목 설정
plt.title('Model MSE')

# 3) X축, Y축 라벨 설정
plt.xlabel('Epochs')
plt.ylabel('MSE')

# 4) 범례(Legend) 설정: 입력한 plot 순서대로 'mse', 'val_mse'로 표시
plt.legend(['mse', 'val_mse'])

# 5) 그래프 출력
plt.show()


---

## 3. 코드 핵심 구성요소 상세 해설

| 구문 | 역할 및 가이드 요구사항 반영 |
| :--- | :--- |
| `history.history` | `model.fit()` 결과가 담긴 변수로, 에포크별 `mse`, `val_mse`, `loss` 등의 기록이 파이썬 딕셔너리 형태로 저장되어 있습니다. |
| `plt.plot(...)` 두 번 연속 호출 | 두 개의 선을 하나의 그래프 영역 위에 겹쳐서 그립니다. |
| `plt.title('Model MSE')` | 그래프 최상단 타이틀을 지시사항에 맞게 `'Model MSE'`로 설정합니다. |
| `plt.xlabel('Epochs')` | X축 이름을 지시사항 요구에 맞게 `'Epochs'`로 표기합니다. |
| `plt.ylabel('MSE')` | Y축 이름을 `'MSE'`로 표기합니다. |
| `plt.legend(['mse', 'val_mse'])` | 첫 번째 그린 선(`mse`)과 두 번째 그린 선(`val_mse`)에 해당하는 범례 상자를 만듭니다. |
